In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import glob
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import csv
import os
import datetime

In [ ]:

# ============================================================================
# HELPER FUNCTIONS FOR CUDA DETECTION AND DEVICE MANAGEMENT
# ============================================================================

def setup_device():
    """
    Detect available compute devices and select the best one
    Prioritizes CUDA GPU if available, falls back to CPU

    Returns:
        device: torch.device object configured for the best available device
        device_name: string description of the selected device
    """
    # Check if CUDA is available on the system
    if torch.cuda.is_available():
        # CUDA is available, select GPU device
        device = torch.device('cuda')
        # Get the name of the GPU device
        device_name = torch.cuda.get_device_name(0)
        # Get GPU memory information
        gpu_memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert to GB

        print("\n" + "="*70)
        print("CUDA DEVICE DETECTED - GPU ACCELERATION ENABLED")
        print("="*70)
        print(f"✓ Device: {device_name}")
        print(f"✓ Compute Capability: {torch.cuda.get_device_capability(0)}")
        print(f"✓ Total Memory: {gpu_memory_total:.2f} GB")
        print(f"✓ PyTorch Version: {torch.__version__}")
        print(f"✓ CUDA Version: {torch.version.cuda}")
        print("="*70 + "\n")

        return device, f"{device_name} (GPU)"
    else:
        # CUDA not available, fall back to CPU
        device = torch.device('cpu')
        print("\n" + "="*70)
        print("CUDA NOT AVAILABLE - USING CPU")
        print("="*70)
        print("⚠ GPU not detected. Using CPU for training (slower)")
        print(f"✓ CPU Cores: {os.cpu_count()}")
        print("="*70 + "\n")

        return device, "CPU"

def extract_id_from_filename(filename):
    """
    Extract ID from filename (e.g., 'HVC224.92+6.10+100.txt' -> 'HVC224.92+6.10+100')

    Args:
        filename: full path to file

    Returns:
        ID string without extension
    """
    # Get the base filename without directory path
    basename = os.path.basename(filename)
    # Remove .txt extension
    return os.path.splitext(basename)[0]

def create_labels_from_ids(file_list, false_ids_list=None):
    """
    Create labels based on file IDs
    If ID is in error list, label is False (0), otherwise True (1)

    Args:
        file_list: list of file paths
        false_ids_list: list of IDs that should be labeled as False

    Returns:
        tuple: (labels_array, ids_list)
            - labels_array: numpy array of binary labels
            - ids_list: list of ID strings
    """
    # Default false IDs if not provided
    if false_ids_list is None:
        false_ids_list = []

    # Extract IDs from file paths
    ids = [extract_id_from_filename(f) for f in file_list]

    # Create binary labels: 0 if in false list, 1 otherwise
    labels = np.array([0 if id_str in false_ids_list else 1 for id_str in ids])

    return labels, ids

def pad_or_trim_spectrum(spec, target_len):
    """
    Pad spectrum with zeros on the right or trim to reach target length

    Args:
        spec: input spectrum array
        target_len: target length

    Returns:
        spectrum array with target length
    """
    if len(spec) < target_len:
        # Pad with zeros on the right
        return np.pad(spec, (0, target_len - len(spec)), mode='constant', constant_values=0)
    elif len(spec) > target_len:
        # Trim to target length
        return spec[:target_len]
    else:
        return spec

In [ ]:
# ============================================================================
# DEFINE FALSE CANDIDATES LISTS
# ============================================================================

# False candidates (4 samples)
false_candidates = [
    # Test 1
    'HVC225.14+6.52+100', 'HVC224.92+6.10+100', 'HVC228.40+12.31+106', 'HVC228.83+13.02+106',
    # Test 2
    'HVC199.50-31.25--58', 'HVC197.79-33.79--56', 'HVC196.17-36.17--74', 'HVC199.27-31.79--55', 'HVC196.58-35.58--87', 'HVC202.58-26.54--82', 'HVC197.20-34.63--56', 'HVC199.10-31.75--57', 'HVC197.55-34.15--71', 'HVC199.71-29.60--223'
]

In [ ]:
# ============================================================================
# SECTION 1: 2D IMAGE TRAINING AND EVALUATION
# ============================================================================

print("\n" + "="*70)
print("HVC CATALOG CNN MODEL - 2D & 1D COMBINED CLASSIFICATION")
print("="*70)
print(f"Training Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"User: Firestar-Reimu")

# Setup CUDA device
device, device_name = setup_device()

print("="*70)
print("LOADING AND SPLITTING DATA (2D IMAGES)")
print("="*70)

# 1. Load all 2D arrays from ./2D_morph/
all_files_2d = sorted(glob.glob('./2D_morph/*.txt'))

# Split files into training (even indices) and testing (odd indices)
file_list_2d_train = [f for i, f in enumerate(all_files_2d) if i % 2 == 0]
file_list_2d_test = [f for i, f in enumerate(all_files_2d) if i % 2 != 0]

arrays_2d_train = [np.loadtxt(fname) for fname in file_list_2d_train]
arrays_2d_test = [np.loadtxt(fname) for fname in file_list_2d_test]

# Create labels for training and test sets using the same false candidates list
labels_2d_train, ids_2d_train = create_labels_from_ids(file_list_2d_train, false_candidates)
labels_2d_test, ids_2d_test = create_labels_from_ids(file_list_2d_test, false_candidates)

print(f"✓ Total 2D samples: {len(all_files_2d)}")
print(f"✓ Training samples (even indices): {len(arrays_2d_train)}")
print(f"  - False: {len([l for l in labels_2d_train if l == 0])}")
print(f"  - True: {len([l for l in labels_2d_train if l == 1])}")

print(f"\n✓ Test samples (odd indices): {len(arrays_2d_test)}")
print(f"  - False: {len([l for l in labels_2d_test if l == 0])}")
print(f"  - True: {len([l for l in labels_2d_test if l == 1])}")

# %%
# 3. Define Dataset class for 2D arrays
class VariableShapeMorphDataset(Dataset):
    """
    Dataset class for loading 2D morphological arrays with variable shapes
    This class inherits from PyTorch's Dataset base class to enable use with DataLoader
    """

    def __init__(self, arrays, labels):
        """
        Initialize the dataset with arrays and their corresponding labels

        Args:
            arrays: list of 2D numpy arrays (each array can have different shape)
            labels: list of binary labels (0 or 1) corresponding to each array
        """
        self.arrays = arrays
        self.labels = labels

    def __len__(self):
        """Return the total number of samples in the dataset"""
        return len(self.arrays)

    def __getitem__(self, idx):
        """
        Retrieve a single sample and its label by index

        Args:
            idx: integer index of the sample to retrieve

        Returns:
            tuple: (input_tensor, label_tensor)
        """
        # Convert to tensor and add channel dimension: (1, H, W)
        x = torch.tensor(self.arrays[idx], dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

# Create train and test datasets for 2D
train_dataset_2d = VariableShapeMorphDataset(arrays_2d_train, labels_2d_train)
test_dataset_2d = VariableShapeMorphDataset(arrays_2d_test, labels_2d_test)
train_loader_2d = DataLoader(train_dataset_2d, batch_size=1, shuffle=True)
test_loader_2d = DataLoader(test_dataset_2d, batch_size=1, shuffle=False)

# %%
# 4. Define 2D CNN model (supports variable input shapes)
class VariableShapeCNN(nn.Module):
    """
    2D Convolutional Neural Network with fully convolutional architecture
    Supports arbitrary input spatial dimensions
    """

    def __init__(self):
        super().__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        # Global Average Pooling to handle variable input sizes
        self.gap = nn.AdaptiveAvgPool2d(1)
        # Fully connected layer for binary classification
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        """
        Forward pass through the network

        Args:
            x: input tensor of shape (batch_size, 1, H, W)

        Returns:
            output: tensor of shape (batch_size,) with sigmoid probabilities
        """
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # Global average pooling
        x = self.gap(x)
        # Reshape to (batch, 16)
        x = x.view(x.size(0), -1)
        # Output with sigmoid activation
        x = torch.sigmoid(self.fc(x)).squeeze(-1)
        return x

# Create model and move to device (GPU or CPU)
model_2d = VariableShapeCNN().to(device)
optimizer_2d = torch.optim.Adam(model_2d.parameters(), lr=1e-3)
loss_fn_2d = nn.BCELoss()

print(f"\n2D Model created and moved to device: {device_name}")
print(f"Total 2D model parameters: {sum(p.numel() for p in model_2d.parameters()):,}")

# %%
# 5. Train 2D model
num_epochs_2d = 50
print("\n" + "="*70)
print(f"TRAINING 2D MODEL ({num_epochs_2d} epochs)")
print("="*70)

for epoch in range(num_epochs_2d):
    model_2d.train()
    losses = []

    for x, y in train_loader_2d:
        # Move batch data to device (GPU or CPU)
        x, y = x.to(device), y.to(device)

        out = model_2d(x)
        loss = loss_fn_2d(out.view(-1), y.view(-1))
        optimizer_2d.zero_grad()
        loss.backward()
        optimizer_2d.step()
        losses.append(loss.item())

    # MODIFICATION 2: Print train loss every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = np.mean(losses)
        print(f'Epoch {epoch+1:3d}/{num_epochs_2d}, avg train loss: {avg_loss:.6f}')

# %%
# 6. Evaluate 2D model on test set
print("\n" + "="*70)
print("EVALUATING 2D MODEL ON TEST SET")
print("="*70)

model_2d.eval()
preds_2d = []
trues_2d = []

with torch.no_grad():
    for x, y in test_loader_2d:
        # Move batch data to device
        x, y = x.to(device), y.to(device)

        out = model_2d(x)
        pred = (out.view(-1) > 0.5).int().item()
        preds_2d.append(pred)
        trues_2d.append(y.item())

# Print overall metrics for 2D model
print('\n=== 2D Model Test Results ===')
acc_2d = accuracy_score(trues_2d, preds_2d)
prec_2d = precision_score(trues_2d, preds_2d, zero_division=0)
recall_2d = recall_score(trues_2d, preds_2d, zero_division=0)
f1_2d = f1_score(trues_2d, preds_2d, zero_division=0)

print('Test Accuracy:', acc_2d)
print('Test Precision:', prec_2d)
print('Test Recall:', recall_2d)
print('Test F1:', f1_2d)

# Save 2D model weights to file
torch.save(model_2d.state_dict(), "cnn_2d_morph.pt")
print("\n2D Model saved to: cnn_2d_morph.pt")


HVC CATALOG CNN MODEL - 2D & 1D COMBINED CLASSIFICATION
Training Date: 2025-11-13 10:53:53
User: Firestar-Reimu

CUDA DEVICE DETECTED - GPU ACCELERATION ENABLED
✓ Device: NVIDIA GeForce RTX 4060 Ti
✓ Compute Capability: (8, 9)
✓ Total Memory: 8.15 GB
✓ PyTorch Version: 2.7.1
✓ CUDA Version: 12.9

LOADING AND SPLITTING DATA (2D IMAGES)
✓ Total 2D samples: 38
✓ Training samples (even indices): 19
  - False: 0
  - True: 19

✓ Test samples (odd indices): 19
  - False: 0
  - True: 19

2D Model created and moved to device: NVIDIA GeForce RTX 4060 Ti (GPU)
Total 2D model parameters: 1,265

TRAINING 2D MODEL (50 epochs)
Epoch   5/50, avg train loss: 0.017316
Epoch  10/50, avg train loss: 0.003251
Epoch  15/50, avg train loss: 0.001047
Epoch  20/50, avg train loss: 0.000429
Epoch  25/50, avg train loss: 0.000229
Epoch  30/50, avg train loss: 0.000140
Epoch  35/50, avg train loss: 0.000092
Epoch  40/50, avg train loss: 0.000066
Epoch  45/50, avg train loss: 0.000049
Epoch  50/50, avg train loss

In [ ]:
# ============================================================================
# SECTION 2: 1D SPECTRUM TRAINING AND EVALUATION
# ============================================================================

print("\n" + "="*70)
print("LOADING AND SPLITTING DATA (1D SPECTRUM)")
print("="*70)

# 1. Load 1D spectrum and reference spectrum files
all_spectrum_files = sorted(glob.glob('./crafts_spectrum/*.txt'))
all_ref_spectrum_files = sorted(glob.glob('./hi4pi_spectrum/*.txt'))

# Split files into training (even) and testing (odd) sets
spectrum_files_train = [f for i, f in enumerate(all_spectrum_files) if i % 2 == 0]
spectrum_files_test = [f for i, f in enumerate(all_spectrum_files) if i % 2 != 0]
ref_spectrum_files_train = [f for i, f in enumerate(all_ref_spectrum_files) if i % 2 == 0]
ref_spectrum_files_test = [f for i, f in enumerate(all_ref_spectrum_files) if i % 2 != 0]

spectra_1d_train = [np.loadtxt(fname) for fname in spectrum_files_train]
spectra_1d_test = [np.loadtxt(fname) for fname in spectrum_files_test]
spectra_1d_ref_train = [np.loadtxt(fname) for fname in ref_spectrum_files_train]
spectra_1d_ref_test = [np.loadtxt(fname) for fname in ref_spectrum_files_test]

# Create labels for training and test sets
labels_1d_train, ids_1d_train = create_labels_from_ids(spectrum_files_train, false_candidates)
labels_1d_test, ids_1d_test = create_labels_from_ids(spectrum_files_test, false_candidates)

print(f"✓ Total 1D samples: {len(all_spectrum_files)}")
print(f"✓ Training samples (even indices): {len(spectra_1d_train)}")
print(f"  - False: {len([l for l in labels_1d_train if l == 0])}")
print(f"  - True: {len([l for l in labels_1d_train if l == 1])}")

print(f"\n✓ Test samples (odd indices): {len(spectra_1d_test)}")
print(f"  - False: {len([l for l in labels_1d_test if l == 0])}")
print(f"  - True: {len([l for l in labels_1d_test if l == 1])}")

# %%
# 3. Normalize spectrum lengths
# Find the max length across both train and test for consistent padding
all_spectra = spectra_1d_train + spectra_1d_test + spectra_1d_ref_train + spectra_1d_ref_test
target_length = max([len(s) for s in all_spectra])

print(f"\nNormalization settings:")
print(f"  Target length for all spectra (max found): {target_length}")

# Normalize all spectra to the same target length
spectra_1d_train = [pad_or_trim_spectrum(s, target_length) for s in spectra_1d_train]
spectra_1d_ref_train = [pad_or_trim_spectrum(s, target_length) for s in spectra_1d_ref_train]
spectra_1d_test = [pad_or_trim_spectrum(s, target_length) for s in spectra_1d_test]
spectra_1d_ref_test = [pad_or_trim_spectrum(s, target_length) for s in spectra_1d_ref_test]

print(f"✓ All training spectra normalized to: {len(spectra_1d_train[0]) if spectra_1d_train else 'N/A'}")
print(f"✓ All test spectra normalized to: {len(spectra_1d_test[0]) if spectra_1d_test else 'N/A'}")

# %%
# 4. Define Dataset class for 1D spectra
class Spectrum1DDataset(Dataset):
    """Dataset class for 1D spectra with reference spectra"""

    def __init__(self, spectra, spectra_ref, labels):
        self.spectra = spectra
        self.spectra_ref = spectra_ref
        self.labels = labels

    def __len__(self):
        return len(self.spectra)

    def __getitem__(self, idx):
        """
        Retrieve a single sample spectrum pair and its label by index

        Returns:
            tuple: (input_tensor, label_tensor)
                   where input_tensor is shape (2, L) containing both spectra
        """
        # Stack the two spectra along axis 0 to create a 2-channel input
        x = torch.tensor(np.stack([self.spectra[idx], self.spectra_ref[idx]], axis=0),
                        dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

# Create train and test datasets for 1D
train_dataset_1d = Spectrum1DDataset(spectra_1d_train, spectra_1d_ref_train, labels_1d_train)
test_dataset_1d = Spectrum1DDataset(spectra_1d_test, spectra_1d_ref_test, labels_1d_test)
train_loader_1d = DataLoader(train_dataset_1d, batch_size=1, shuffle=True)
test_loader_1d = DataLoader(test_dataset_1d, batch_size=1, shuffle=False)

# %%
# 5. Define 1D CNN model for spectrum classification
spectrum_length_train = target_length

class CNN1D(nn.Module):
    """
    1D Convolutional Neural Network for spectral data
    Takes two channels: current spectrum and reference spectrum
    """

    def __init__(self, length):
        super().__init__()
        # 从kernel_size=5 改为 kernel_size=11
        self.conv1 = nn.Conv1d(2, 8, kernel_size=11, padding=5)
        self.pool = nn.MaxPool1d(2)

        # Conv2保持kernel_size=5，但padding增加
        self.conv2 = nn.Conv1d(8, 16, kernel_size=7, padding=3)
        self.pool2 = nn.MaxPool1d(2)

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        """
        Forward pass through the network

        Args:
            x: input tensor of shape (batch_size, 2, L)

        Returns:
            output: tensor of shape (batch_size,) with sigmoid probabilities
        """
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        # Global average pooling
        x = self.gap(x)
        # Reshape to (batch, 16)
        x = x.view(x.size(0), -1)
        # Output with sigmoid activation
        x = torch.sigmoid(self.fc(x)).squeeze(-1)
        return x

# Create model and move to device (GPU or CPU)
model_1d = CNN1D(spectrum_length_train).to(device)
optimizer_1d = torch.optim.Adam(model_1d.parameters(), lr=1e-3)
loss_fn_1d = nn.BCELoss()

print(f"\n1D Model created and moved to device: {device_name}")
print(f"Total 1D model parameters: {sum(p.numel() for p in model_1d.parameters()):,}")

# %%
# 6. Train 1D model
num_epochs_1d = 100
print("\n" + "="*70)
print(f"TRAINING 1D MODEL ({num_epochs_1d} epochs)")
print("="*70)

for epoch in range(num_epochs_1d):
    model_1d.train()
    losses = []

    for x, y in train_loader_1d:
        # Move batch data to device (GPU or CPU)
        x, y = x.to(device), y.to(device)

        out = model_1d(x)
        loss = loss_fn_1d(out.view(-1), y.view(-1))
        optimizer_1d.zero_grad()
        loss.backward()
        optimizer_1d.step()
        losses.append(loss.item())

    # MODIFICATION 2: Print train loss every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = np.mean(losses)
        print(f'Epoch {epoch+1:3d}/{num_epochs_1d}, avg train loss: {avg_loss:.6f}')

# %%
# 7. Evaluate 1D model on test set
print("\n" + "="*70)
print("EVALUATING 1D MODEL ON TEST SET")
print("="*70)

model_1d.eval()
preds_1d = []
trues_1d = []

with torch.no_grad():
    for x, y in test_loader_1d:
        # Move batch data to device
        x, y = x.to(device), y.to(device)

        out = model_1d(x)
        pred = (out.view(-1) > 0.5).int().item()
        preds_1d.append(pred)
        trues_1d.append(y.item())

# Print overall metrics for 1D model
print('\n=== 1D Model Test Results ===')
acc_1d = accuracy_score(trues_1d, preds_1d)
prec_1d = precision_score(trues_1d, preds_1d, zero_division=0)
recall_1d = recall_score(trues_1d, preds_1d, zero_division=0)
f1_1d = f1_score(trues_1d, preds_1d, zero_division=0)

print('Test Accuracy:', acc_1d)
print('Test Precision:', prec_1d)
print('Test Recall:', recall_1d)
print('Test F1:', f1_1d)

# Save 1D model weights to file
torch.save(model_1d.state_dict(), "cnn_1d_spectrum.pt")
print("\n1D Model saved to: cnn_1d_spectrum.pt")


LOADING AND SPLITTING DATA (1D SPECTRUM)
✓ Total 1D samples: 38
✓ Training samples (even indices): 19
  - False: 0
  - True: 19

✓ Test samples (odd indices): 19
  - False: 0
  - True: 19

Normalization settings:
  Target length for all spectra (max found): 79
✓ All training spectra normalized to: 79
✓ All test spectra normalized to: 79

1D Model created and moved to device: NVIDIA GeForce RTX 4060 Ti (GPU)
Total 1D model parameters: 1,113

TRAINING 1D MODEL (100 epochs)
Epoch   5/100, avg train loss: 0.173766
Epoch  10/100, avg train loss: 0.013794
Epoch  15/100, avg train loss: 0.005345
Epoch  20/100, avg train loss: 0.002903
Epoch  25/100, avg train loss: 0.001825
Epoch  30/100, avg train loss: 0.001269
Epoch  35/100, avg train loss: 0.000922
Epoch  40/100, avg train loss: 0.000698
Epoch  45/100, avg train loss: 0.000550
Epoch  50/100, avg train loss: 0.000442
Epoch  55/100, avg train loss: 0.000361
Epoch  60/100, avg train loss: 0.000302
Epoch  65/100, avg train loss: 0.000254
Epo

In [ ]:
# ============================================================================
# SECTION 3: COMBINED PREDICTIONS (2D AND 1D)
# ============================================================================

print("\n" + "="*70)
print("GENERATING COMBINED PREDICTIONS")
print("="*70)

# Combine 2D and 1D predictions
# Only mark as True (signal) if BOTH 2D and 1D predict True
# First ensure both test sets have matching IDs
assert ids_2d_test == ids_1d_test, "2D and 1D test sets must have matching IDs"

combined_preds = []
combined_labels = []
combined_ids = []

for id_str, pred_2d, pred_1d, true_label in zip(ids_2d_test, preds_2d, preds_1d, trues_2d):
    # Combined prediction: True only if both 2D AND 1D predict True
    combined_pred = 1 if (pred_2d == 1 and pred_1d == 1) else 0
    combined_preds.append(combined_pred)
    combined_labels.append(true_label)
    combined_ids.append(id_str)

# Calculate combined metrics
print('\n=== Combined Model Test Results ===')
acc_combined = accuracy_score(combined_labels, combined_preds)
prec_combined = precision_score(combined_labels, combined_preds, zero_division=0)
recall_combined = recall_score(combined_labels, combined_preds, zero_division=0)
f1_combined = f1_score(combined_labels, combined_preds, zero_division=0)

print('Test Accuracy:', acc_combined)
print('Test Precision:', prec_combined)
print('Test Recall:', recall_combined)
print('Test F1:', f1_combined)


GENERATING COMBINED PREDICTIONS

=== Combined Model Test Results ===
Test Accuracy: 1.0
Test Precision: 1.0
Test Recall: 1.0
Test F1: 1.0


In [ ]:
# ============================================================================
# SECTION 4: OUTPUT DETAILED RESULTS
# ============================================================================

print("\n" + "="*70)
print("SAVING DETAILED PREDICTIONS TO CSV")
print("="*70)

# Define output file name
output_file_combined = "combined_predictions_by_label.csv"

# Write combined predictions grouped by label ID to CSV file
with open(output_file_combined, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Write header row
    writer.writerow(['Label_ID', '2D_Prediction', '1D_Prediction', 'Combined_Prediction',
                     'True_Label', '2D_Match', '1D_Match', 'Combined_Match'])

    # Write prediction for each label
    for idx, (id_str, pred_2d, pred_1d, combined_pred, true_label) in enumerate(
            zip(combined_ids, preds_2d, preds_1d, combined_preds, combined_labels)):

        # Convert predictions to string format
        pred_2d_str = 'Signal' if pred_2d == 1 else 'Noise'
        pred_1d_str = 'Signal' if pred_1d == 1 else 'Noise'
        combined_pred_str = 'Signal' if combined_pred == 1 else 'Noise'
        true_label_str = 'Signal' if true_label == 1 else 'Noise'

        # Check if each prediction matches ground truth
        match_2d = 'Yes' if pred_2d == true_label else 'No'
        match_1d = 'Yes' if pred_1d == true_label else 'No'
        match_combined = 'Yes' if combined_pred == true_label else 'No'

        # Write row
        writer.writerow([id_str, pred_2d_str, pred_1d_str, combined_pred_str,
                        true_label_str, match_2d, match_1d, match_combined])

print(f"Combined predictions saved to: {output_file_combined}")

# %%
# Print summary table to console
print("\n" + "="*70)
print("DETAILED PREDICTIONS BY LABEL")
print("="*70)
print(f"{'Label_ID':<25} {'2D':<10} {'1D':<10} {'Combined':<10} {'True':<10} {'Status':<20}")
print("-" * 90)

for id_str, pred_2d, pred_1d, combined_pred, true_label in zip(
        combined_ids, preds_2d, preds_1d, combined_preds, combined_labels):

    pred_2d_str = 'Signal' if pred_2d == 1 else 'Noise'
    pred_1d_str = 'Signal' if pred_1d == 1 else 'Noise'
    combined_pred_str = 'Signal' if combined_pred == 1 else 'Noise'
    true_label_str = 'Signal' if true_label == 1 else 'Noise'

    # Show if combined prediction is correct
    status = '✓ CORRECT' if combined_pred == true_label else '✗ WRONG'

    print(f"{id_str:<25} {pred_2d_str:<10} {pred_1d_str:<10} {combined_pred_str:<10} "
          f"{true_label_str:<10} {status:<20}")


SAVING DETAILED PREDICTIONS TO CSV
Combined predictions saved to: combined_predictions_by_label.csv

DETAILED PREDICTIONS BY LABEL
Label_ID                  2D         1D         Combined   True       Status              
------------------------------------------------------------------------------------------
HVC196.17-36.17-75        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC196.58-35.58-89        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC197.37-31.87-129       Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC197.79-33.79-57        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC199.18-30.35-58        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC199.50-31.25-62        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC200.19-33.12-63        Signal     Signal     Signal     Signal     ✓ CORRECT           
HVC203.39-23.27-82        Signal     Signal     S

In [ ]:
# ============================================================================
# SECTION 5: PREDICTION STATISTICS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION STATISTICS")
print("="*70)

# Calculate 2D model statistics
# NEW: Count correct and wrong predictions for 2D model
pred_2d_correct = sum(1 for pred, true in zip(preds_2d, trues_2d) if pred == true)
pred_2d_wrong = sum(1 for pred, true in zip(preds_2d, trues_2d) if pred != true)
pred_2d_total = len(preds_2d)
pred_2d_correct_pct = (pred_2d_correct / pred_2d_total * 100) if pred_2d_total > 0 else 0
pred_2d_wrong_pct = (pred_2d_wrong / pred_2d_total * 100) if pred_2d_total > 0 else 0

print("\n=== 2D MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_2d_total}")
print(f"  ✓ CORRECT: {pred_2d_correct:3d} ({pred_2d_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_2d_wrong:3d} ({pred_2d_wrong_pct:6.2f}%)")

# Calculate 1D model statistics
# NEW: Count correct and wrong predictions for 1D model
pred_1d_correct = sum(1 for pred, true in zip(preds_1d, trues_1d) if pred == true)
pred_1d_wrong = sum(1 for pred, true in zip(preds_1d, trues_1d) if pred != true)
pred_1d_total = len(preds_1d)
pred_1d_correct_pct = (pred_1d_correct / pred_1d_total * 100) if pred_1d_total > 0 else 0
pred_1d_wrong_pct = (pred_1d_wrong / pred_1d_total * 100) if pred_1d_total > 0 else 0

print("\n=== 1D MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_1d_total}")
print(f"  ✓ CORRECT: {pred_1d_correct:3d} ({pred_1d_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_1d_wrong:3d} ({pred_1d_wrong_pct:6.2f}%)")

# Calculate combined model statistics
# NEW: Count correct and wrong predictions for combined model
pred_combined_correct = sum(1 for pred, true in zip(combined_preds, combined_labels) if pred == true)
pred_combined_wrong = sum(1 for pred, true in zip(combined_preds, combined_labels) if pred != true)
pred_combined_total = len(combined_preds)
pred_combined_correct_pct = (pred_combined_correct / pred_combined_total * 100) if pred_combined_total > 0 else 0
pred_combined_wrong_pct = (pred_combined_wrong / pred_combined_total * 100) if pred_combined_total > 0 else 0

print("\n=== COMBINED MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_combined_total}")
print(f"  ✓ CORRECT: {pred_combined_correct:3d} ({pred_combined_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_combined_wrong:3d} ({pred_combined_wrong_pct:6.2f}%)")

# Print comparison summary
print("\n" + "="*70)
print("MODELS COMPARISON SUMMARY")
print("="*70)
print(f"{'Model':<20} {'Accuracy':<15} {'Correct':<20} {'Wrong':<20}")
print("-" * 75)
print(f"{'2D Model':<20} {acc_2d*100:6.2f}%        {pred_2d_correct:3d}/{pred_2d_total:3d} ({pred_2d_correct_pct:5.2f}%) {pred_2d_wrong:3d}/{pred_2d_total:3d} ({pred_2d_wrong_pct:5.2f}%)")
print(f"{'1D Model':<20} {acc_1d*100:6.2f}%        {pred_1d_correct:3d}/{pred_1d_total:3d} ({pred_1d_correct_pct:5.2f}%) {pred_1d_wrong:3d}/{pred_1d_total:3d} ({pred_1d_wrong_pct:5.2f}%)")
print(f"{'Combined Model':<20} {acc_combined*100:6.2f}%        {pred_combined_correct:3d}/{pred_combined_total:3d} ({pred_combined_correct_pct:5.2f}%) {pred_combined_wrong:3d}/{pred_combined_total:3d} ({pred_combined_wrong_pct:5.2f}%)")


PREDICTION STATISTICS

=== 2D MODEL PREDICTIONS ===
Total predictions: 19
  ✓ CORRECT:  19 (100.00%)
  ✗ WRONG:     0 (  0.00%)

=== 1D MODEL PREDICTIONS ===
Total predictions: 19
  ✓ CORRECT:  19 (100.00%)
  ✗ WRONG:     0 (  0.00%)

=== COMBINED MODEL PREDICTIONS ===
Total predictions: 19
  ✓ CORRECT:  19 (100.00%)
  ✗ WRONG:     0 (  0.00%)

MODELS COMPARISON SUMMARY
Model                Accuracy        Correct              Wrong               
---------------------------------------------------------------------------
2D Model             100.00%         19/ 19 (100.00%)   0/ 19 ( 0.00%)
1D Model             100.00%         19/ 19 (100.00%)   0/ 19 ( 0.00%)
Combined Model       100.00%         19/ 19 (100.00%)   0/ 19 ( 0.00%)


In [ ]:
# ============================================================================
# FINAL CLEANUP AND SUMMARY
# ============================================================================

print("\n" + "="*70)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Completion Time (UTC): {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"User: Firestar-Reimu")
print(f"Device Used: {device_name}")
print(f"Total Epochs 2D: {num_epochs_2d}, Total Epochs 1D: {num_epochs_1d}")
print(f"Loss Print Interval: Every 5 epochs")
print(f"\nTraining Summary:")
print(f"  2D training samples: {len(arrays_2d_train)}")
print(f"    - False: {len([l for l in labels_2d_train if l == 0])}")
print(f"    - True: {len([l for l in labels_2d_train if l == 1])}")
print(f"  1D training samples: {len(spectra_1d_train)}")
print(f"    - False: {len([l for l in labels_1d_train if l == 0])}")
print(f"    - True: {len([l for l in labels_1d_train if l == 1])}")
print(f"\nTest Summary:")
print(f"  2D test samples: {len(arrays_2d_test)}")
print(f"    - False: {len([l for l in labels_2d_test if l == 0])}")
print(f"    - True: {len([l for l in labels_2d_test if l == 1])}")
print(f"  1D test samples: {len(spectra_1d_test)}")
print(f"    - False: {len([l for l in labels_1d_test if l == 0])}")
print(f"    - True: {len([l for l in labels_1d_test if l == 1])}")
print(f"\nModel Performance:")
print(f"  2D Accuracy:       {acc_2d*100:6.2f}% ({pred_2d_correct}/{pred_2d_total})")
print(f"  1D Accuracy:       {acc_1d*100:6.2f}% ({pred_1d_correct}/{pred_1d_total})")
print(f"  Combined Accuracy: {acc_combined*100:6.2f}% ({pred_combined_correct}/{pred_combined_total})")
print(f"\nModel Files:")
print(f"  2D Model weights: cnn_2d_morph.pt")
print(f"  1D Model weights: cnn_1d_spectrum.pt")
print(f"\nResults:")
print(f"  Predictions CSV: {output_file_combined}")
print("="*70 + "\n")

# Clear GPU memory if CUDA was used
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU memory cleared")


TRAINING COMPLETED SUCCESSFULLY
Completion Time (UTC): 2025-11-13 10:53:56
User: Firestar-Reimu
Device Used: NVIDIA GeForce RTX 4060 Ti (GPU)
Total Epochs 2D: 50, Total Epochs 1D: 100
Loss Print Interval: Every 5 epochs

Training Summary:
  2D training samples: 19
    - False: 0
    - True: 19
  1D training samples: 19
    - False: 0
    - True: 19

Test Summary:
  2D test samples: 19
    - False: 0
    - True: 19
  1D test samples: 19
    - False: 0
    - True: 19

Model Performance:
  2D Accuracy:       100.00% (19/19)
  1D Accuracy:       100.00% (19/19)
  Combined Accuracy: 100.00% (19/19)

Model Files:
  2D Model weights: cnn_2d_morph.pt
  1D Model weights: cnn_1d_spectrum.pt

Results:
  Predictions CSV: combined_predictions_by_label.csv

✓ GPU memory cleared
